In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import(
accuracy_score,
f1_score,
precision_score,
recall_score,
confusion_matrix,
roc_auc_score)
from sklearn.model_selection import GridSearchCV ,cross_val_score,cross_val_predict

In [2]:
df = pd.read_csv('appointments_clean.csv')

In [3]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [4]:
df['reminder_sent'] = df['reminder_sent'].map({'Yes': 1,'No':0})

In [5]:
selected_feat = [
    'age',
    'distance_km',
    'fee_inr',
    'booking_channel',
    'visit_type',
    'reminder_sent',	
    'prior_no_shows',
    'lead_days'
]

In [6]:
X_base = df[selected_feat].copy()
y = df['no_show']

In [7]:
X = pd.get_dummies(X_base, columns=["booking_channel","visit_type"],dtype=float)


In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size= 0.2,random_state = 42)

In [9]:
print(f'training set size : {X_train.shape}')
print(f'test set size : {X_test.shape}')
print(f'No. of no-shows in test set :{y_test.sum()}')


training set size : (792, 11)
test set size : (198, 11)
No. of no-shows in test set :49


In [10]:
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline Accuracy of always predicting 'Attended' :{baseline_accuracy:.3f}")

Baseline Accuracy of always predicting 'Attended' :0.753


In [11]:
X_test.dtypes

age                          int64
distance_km                float64
fee_inr                      int64
reminder_sent                int64
prior_no_shows               int64
lead_days                    int64
booking_channel_App        float64
booking_channel_Phone      float64
booking_channel_Walk-In    float64
visit_type_Consultation    float64
visit_type_Follow-up       float64
dtype: object

In [12]:
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test) 

In [13]:
lr_model = LogisticRegression(max_iter = 1000,random_state =42)
lr_model.fit(X_train_s,y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [14]:
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:,1]

In [15]:
print("--- Logistic Regression Test Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.2f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_lr):.2f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

--- Logistic Regression Test Metrics ---
Accuracy:  0.79
Precision: 0.63
Recall:    0.35
F1 Score:  0.45
ROC-AUC:   0.79
Confusion Matrix:
 [[139  10]
 [ 32  17]]


In [16]:
print("\n--- Feature Coefficients ---")
for col, coef in zip(X.columns, lr_model.coef_[0]):
  print(f"{col}: {coef:.4f}")


--- Feature Coefficients ---
age: -0.2539
distance_km: 0.2534
fee_inr: 0.4528
reminder_sent: -0.5807
prior_no_shows: 0.6496
lead_days: 0.6145
booking_channel_App: -0.0468
booking_channel_Phone: 0.2159
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157


In [17]:
reminder_idx = list(X.columns).index("reminder_sent")
raw_coef = lr_model.coef_[0][reminder_idx]
feature_std = scaler.scale_[reminder_idx]

In [18]:
exp_coef = np.exp(raw_coef)
true_reminder_effect = np.exp(raw_coef / feature_std)

In [19]:
print(
    f"\nRaw model coefficient for reminder_sent: {raw_coef:.2f} -> exp(coef):"
    f" {exp_coef:.2f}"
)
print(
    f"True effect of reminder_sent (divided by feature std {feature_std:.2f}):"
    f" {true_reminder_effect:.2f}"
)


Raw model coefficient for reminder_sent: -0.58 -> exp(coef): 0.56
True effect of reminder_sent (divided by feature std 0.49): 0.31


In [20]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples

In [21]:
train_acc_full = accuracy_score(y_train, dt_model.predict(X_train))
test_acc_full = accuracy_score(y_test, dt_model.predict(X_test))

In [22]:
print(
    f"Unbounded Tree -> Train Acc: {train_acc_full:.2f} | Test Acc:"
    f" {test_acc_full:.2f}"
)

Unbounded Tree -> Train Acc: 1.00 | Test Acc: 0.68


In [23]:
dt_d3_model =DecisionTreeClassifier(max_depth =3,random_state=42)
dt_d3_model.fit(X_train,y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in

In [24]:
train_acc_d3 = accuracy_score(y_train, dt_d3_model.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_d3_model.predict(X_test))

In [25]:
print(
    f"Depth-3 Tree   -> Train Acc: {train_acc_d3:.2f} | Test Acc:"
    f" {test_acc_d3:.2f}"
)

Depth-3 Tree   -> Train Acc: 0.82 | Test Acc: 0.78


In [26]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [27]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

In [28]:
print(f'Random Forest Training Accuracy Score :{accuracy_score(y_train,rf_model.predict(X_train)):.2f}')
print(f'Random Forest Accuracy Score :{accuracy_score(y_test,y_pred_rf):.2f}')
print(f'Random Forest Recall Score :{recall_score(y_test,y_pred_rf):.2f}')
print(f'Random Forest ROC-AUC Score :{roc_auc_score(y_test,y_prob_rf):.2f}')

Random Forest Training Accuracy Score :1.00
Random Forest Accuracy Score :0.74
Random Forest Recall Score :0.27
Random Forest ROC-AUC Score :0.71


In [29]:
importance = pd.Series(rf_model.feature_importances_, index = X.columns)
print('\n Top 3 Important Features :')
print(importance.nlargest(3))


 Top 3 Important Features :
lead_days      0.222369
age            0.201897
distance_km    0.191736
dtype: float64


In [30]:
comparison_data = {
    'Model':['Logistic Regression','3 Depth Decision tree ','Random Forest Classifier'],
    'Accuracy':[accuracy_score(y_test, y_pred_lr),
                test_acc_d3,
                accuracy_score(y_test,rf_model.predict(X_test))
               ],
    'Recall' :[recall_score(y_test, y_pred_lr),
               recall_score(y_test, dt_d3_model.predict(X_test)),
               recall_score(y_test,y_pred_rf)
              ],
    'ROC-AUC':[roc_auc_score(y_test, y_prob_lr),
               roc_auc_score(y_test, dt_d3_model.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,y_prob_rf) 
              ]
}
comparison_df = pd.DataFrame(comparison_data)    
print('\n',comparison_df.to_string(index = False))   


                    Model  Accuracy   Recall  ROC-AUC
     Logistic Regression  0.787879 0.346939 0.788933
  3 Depth Decision tree   0.777778 0.224490 0.699699
Random Forest Classifier  0.742424 0.265306 0.712300


In [31]:
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
    )

grid_search.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 5, ...], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information

In [32]:
print(f'Best Parameter : {grid_search.best_params_}')
print(f'Best ROC-AUC Score :{grid_search.best_score_:.2f}')

Best Parameter : {'max_depth': 5, 'n_estimators': 100}
Best ROC-AUC Score :0.77


In [33]:
lr_pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('clf',LogisticRegression(max_iter=1000))
])

In [34]:
cv_lr = cross_val_score(lr_pipeline,
                        X_train, y_train,
                        cv=5,
                        scoring="roc_auc").mean()

In [52]:
cv_dt_d3 = cross_val_score(DecisionTreeClassifier(max_depth =5,random_state=42),
                           X_train, y_train,
                           cv=5,
                           scoring="roc_auc").mean()

In [36]:
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100,random_state=42),
                        X_train, y_train,
                        cv=5,
                        scoring="roc_auc").mean()

In [37]:
cv_tuned_rf = grid_search.best_score_

In [55]:
print("\n--- 5-Fold Cross-Validated ROC-AUC Comparison ---")
print(f"Logistic Regression: {cv_lr:.2f}")
print(f"Depth-3 Tree:        {cv_dt_d3:.2f}")
print(f"Standard RF:         {cv_rf:.2f}")
print(f"Tuned RF:            {cv_tuned_rf:.2f}")


--- 5-Fold Cross-Validated ROC-AUC Comparison ---
Logistic Regression: 0.77
Depth-3 Tree:        0.70
Standard RF:         0.74
Tuned RF:            0.77


In [39]:
best_model = lr_pipeline
print(best_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000))])


In [40]:
oof_train_probs = cross_val_predict(best_model, X_train, y_train,
                  cv=5,
                  method="predict_proba")[:, 1] 

In [41]:
for thres in [0.5,0.35]:
    pred = (oof_train_probs >= thres).astype(int)
    print(f"\nThreshold: {thres}")
    print(f"  Precision: {precision_score(y_train, pred):.2f}")
    print(f"  Recall:    {recall_score(y_train, pred):.2f}")
    print(f"  F1 Score:  {f1_score(y_train, pred,):.2f}")
    print(f"  Flagged:   {pred.sum()} out of {len(y_train)}")    


Threshold: 0.5
  Precision: 0.58
  Recall:    0.25
  F1 Score:  0.35
  Flagged:   73 out of 792

Threshold: 0.35
  Precision: 0.51
  Recall:    0.47
  F1 Score:  0.49
  Flagged:   154 out of 792


In [42]:
chosen_thres = 0.35

In [43]:
appointment_id = df['appointment_id']

In [44]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, appointment_id,
    test_size=0.2,
    random_state=42
)

In [45]:
best_model.fit(X_train,y_train)
final_test_prob = best_model.predict_proba(X_test)[:,1]
final_pred_test = (final_test_prob >= chosen_thres).astype(int)

In [46]:
print("\n--- Final Test Evaluation (Once Only) ---")
print(f"Precision: {precision_score(y_test, final_pred_test):.2f}")
print(f"Recall:    {recall_score(y_test, final_pred_test):.2f}")
print(f"Flagged:   {final_pred_test.sum()} out of {len(X_test)}")
print(f"ROC-AUC:   {roc_auc_score(y_test, final_test_prob):.2f}")


--- Final Test Evaluation (Once Only) ---
Precision: 0.57
Recall:    0.57
Flagged:   49 out of 198
ROC-AUC:   0.79


In [47]:
test_results_df = pd.DataFrame({
    "appointment_id": appointment_id.loc[X_test.index],
    "no_show": y_test.values,
    "risk_prob": final_test_prob,
    "flag": final_pred_test,
})
test_results_df.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv successfully.")


Saved test_risk_scores.csv successfully.
